# 🧪 CausalCrisis V15: Strategic Research Workspace
Kết hợp nền tảng dữ liệu V2 với đột phá kiến trúc V15.

---

## 🧠 §1 — Knowledge Base: Causal V15 Upgrade
Hệ thống giải quyết 4 điểm nghẽn (B1-B4) bằng kỹ thuật Disentanglement và Monte Carlo Backdoor Adjustment.

## §2 — DATASET & ROBUST INFRASTRUCTURE
Kế thừa các cell quan trọng nhất về quản lý dataset và giải nén từ V2.

In [ ]:
# [CELL QUAN TRỌNG: SETUP & GIẢI NÉN]
!pip install -q open_clip_torch torch_geometric sklearn matplotlib pandas
!apt-get install -y aria2

import os, glob, torch, numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm

RAW_DATA_PATH = '/content/CrisisMMD_v2.0'
DATASET_DIR = '/content/extracted_features'
os.makedirs(DATASET_DIR, exist_ok=True)

def setup_dataset():
    if os.path.exists(RAW_DATA_PATH) and len(glob.glob(f'{RAW_DATA_PATH}/**/*.jpg', recursive=True)) > 10000:
        print('✅ Dataset complete.')
        return
    
    url = 'https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz'
    !aria2c -x 16 -s 16 -k 1M {url}
    !tar -xf CrisisMMD_v2.0.tar.gz && rm CrisisMMD_v2.0.tar.gz
    
    nested = os.path.join(RAW_DATA_PATH, 'CrisisMMD_v2.0')
    if os.path.exists(nested):
        !mv {nested}/* {RAW_DATA_PATH}/ 2>/dev/null || true
        !rmdir {nested} 2>/dev/null || true
    print(f'✅ Extracted to {RAW_DATA_PATH}')

setup_dataset()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## §3 — INTELLIGENT CLIP EXTRACTION
Cell xử lý trích xuất đặc trưng với các cơ chế tìm kiếm đường dẫn ảnh thông minh.

In [ ]:
# [CELL QUAN TRỌNG: EXTRACT FEATURES]
import open_clip
from sklearn.preprocessing import LabelEncoder

def extract_features_v15(data_split='train', task='task2'):
    task_dir = os.path.join(DATASET_DIR, task)
    os.makedirs(task_dir, exist_ok=True)
    
    all_tsvs = glob.glob('/content/**/*.tsv', recursive=True)
    matches = [f for f in all_tsvs if data_split.lower() in f.lower() and 'humanitarian' in f.lower()]
    if not matches: return False
    df = pd.read_csv(matches[0], sep='\t')
    
    global clip_model, preprocess, tokenizer
    if 'clip_model' not in globals():
        print('📥 Loading CLIP (ViT-L-14)...')
        clip_model, _, preprocess = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai")
        clip_model, tokenizer = clip_model.to(device).eval(), open_clip.get_tokenizer("ViT-L-14")
    
    fixed_classes = ['affected_individuals', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'missing_or_found_people', 'not_humanitarian', 'other_relevant_information', 'rescue_volunteering_or_donation_effort', 'vehicle_damage']
    le = LabelEncoder(); le.classes_ = np.array(fixed_classes)
    df = df[df['label_text'].isin(fixed_classes)].copy()
    
    img_feats, txt_feats, labels, domains = [], [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'Encoding {task}/{data_split}'):
        try:
            # Robust Path Matching
            rel_path = str(row['image_path'])
            abs_path = os.path.join(RAW_DATA_PATH, rel_path)
            if not os.path.exists(abs_path):
                alt = os.path.join(RAW_DATA_PATH, rel_path.replace('data_image/', '', 1))
                if os.path.exists(alt): abs_path = alt
                else: continue
            
            with torch.no_grad():
                img_t = preprocess(Image.open(abs_path)).unsqueeze(0).to(device)
                txt_t = tokenizer([str(row['tweet_text'])[:200]]).to(device)
                i_f, t_f = clip_model.encode_image(img_t), clip_model.encode_text(txt_t)
                
            img_feats.append((i_f/i_f.norm(dim=-1, keepdim=True)).cpu().numpy())
            txt_feats.append((t_f/t_f.norm(dim=-1, keepdim=True)).cpu().numpy())
            labels.append(le.transform([row['label_text']])[0])
            domains.append(0)
        except: continue
            
    if img_feats:
        np.save(f'{task_dir}/{data_split}_img.npy', np.vstack(img_feats))
        np.save(f'{task_dir}/{data_split}_txt.npy', np.vstack(txt_feats))
        np.save(f'{task_dir}/{data_split}_labels.npy', np.array(labels))
        np.save(f'{task_dir}/classes.npy', le.classes_)
        return True
    return False

## §4 — V15 CAUSAL ARCHITECTURE & TRAINER
Xây dựng mạng CausalV15 với Disentanglement và hệ thống Backdoor Adjustment.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import classification_report, f1_score

class Disentangler(nn.Module):
    def __init__(self, in_dim=512, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, x): return self.net(x)

def hsic_loss(x, y):
    n = x.size(0); if n < 2: return 0.0
    Kx = torch.exp(-torch.cdist(x, x)**2/10); Ky = torch.exp(-torch.cdist(y, y)**2/10)
    H = torch.eye(n, device=x.device) - 1.0/n
    return torch.trace(Kx @ H @ Ky @ H) / ((n-1)**2)

class CausalV15(nn.Module):
    def __init__(self, n_classes=6):
        super().__init__()
        self.v_proj, self.t_proj = nn.Linear(768, 512), nn.Linear(768, 512)
        self.fuse = nn.Linear(1024, 512)
        self.causal, self.spurious = Disentangler(512, 256), Disentangler(512, 256)
        self.clf = nn.Sequential(nn.Linear(512, 256), nn.GELU(), nn.Linear(256, n_classes))
        
    def forward(self, img, txt, backdoor_xs=None, use_ba=False):
        f = F.gelu(self.fuse(torch.cat([self.v_proj(img), self.t_proj(txt)], -1)))
        xc, xs = self.causal(f), self.spurious(f)
        logits = self.clf(torch.cat([xc, xs*0.1], -1))
        
        logits_ba = None
        if use_ba and backdoor_xs is not None:
            # Monte Carlo BA: E[P(Y|xc, s)] over confounder set
            n_s = backdoor_xs.size(0)
            xc_e = xc.unsqueeze(1).expand(-1, n_s, -1)
            xs_e = backdoor_xs.unsqueeze(0).expand(xc.size(0), -1, -1)
            logits_ba = self.clf(torch.cat([xc_e, xs_e*0.1], -1).view(-1, 512)).view(xc.size(0), n_s, -1).mean(1)
            
        return {'logits': logits, 'logits_ba': logits_ba, 'xc': xc, 'xs': xs}

# [CELL QUAN TRỌNG: TRAIN & EVALUATION]
class TrainerV15:
    def __init__(self, model, lr=5e-4):
        self.model = model.to(device)
        self.opt = torch.optim.AdamW(model.parameters(), lr=lr)
        self.crit = nn.CrossEntropyLoss()
        self.bank = torch.zeros(2000, 256, device=device); self.ptr = 0
        
    def train_epoch(self, loader):
        self.model.train(); total = 0
        for b in loader:
            img, txt, lbl = b[0].to(device), b[1].to(device), b[2].to(device)
            self.opt.zero_grad()
            out = self.model(img, txt)
            loss = self.crit(out['logits'], lbl) + 0.1*hsic_loss(out['xc'], out['xs'])
            loss.backward(); self.opt.step(); total += loss.item()
            # Update memory bank for xs
            size = img.size(0)
            self.bank[self.ptr:self.ptr+size] = out['xs'].detach()[:size]
            self.ptr = (self.ptr + size) % 2000
        return total / len(loader)

def evaluate_v15(model, loader, bank=None, use_ba=False):
    model.eval(); all_probs, all_labels = [], []
    confounders = bank[torch.randint(0, 2000, (100,))] if use_ba else None
    with torch.no_grad():
        for b in loader:
            img, txt, lbl = b[0].to(device), b[1].to(device), b[2].to(device)
            out = model(img, txt, use_ba=use_ba, backdoor_xs=confounders)
            probs = torch.softmax(out['logits_ba'] if use_ba else out['logits'], -1)
            all_probs.append(probs.cpu().numpy()); all_labels.append(lbl.cpu().numpy())
    probs, labels = np.vstack(all_probs), np.concatenate(all_labels)
    preds = np.argmax(probs, axis=1)
    print(classification_report(labels, preds, zero_division=0))
    return probs, labels

## §5 — RESEARCH EXECUTION (LODO & FINAL EVAL)
Khởi chạy lộ trình thí nghiệm OOD.

In [ ]:
# [CELL QUAN TRỌNG: THỰC THI TRAIN]
def run_v15_experiment():
    mapping = {0:0, 1:1, 2:0, 3:0, 4:2, 5:3, 6:4, 7:5} # Merging 8->6
    # 1. Extraction (Uncomment if needed)
    # extract_features_v15('train') 
    # extract_features_v15('dev')
    
    # 2. Data Loading
    # train_loader = create_loader('train', mapping=mapping)
    # val_loader = create_loader('dev', mapping=mapping)
    
    # 3. Model & Training
    model = CausalV15(n_classes=6)
    trainer = TrainerV15(model)
    
    print("🔥 Workspace Ready. Proceed with training loop...")
    pass

run_v15_experiment()